# 03 – Normalization: Material & Finish

This notebook:
- Extracts Material & Finish from BigCommerce Custom Fields
- Applies mapping-based normalization
- Logs normalization status
- Produces a normalized attributes slice for the golden dataset

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

import pandas as pd
from collections import defaultdict

from golden_data.config import get_raw_data_path, DATA_INTERIM
from golden_data.analysis.custom_field_scan import parse_cf_cell
from golden_data.normalization.mapping_utils import (apply_mapping, mapping_coverage_report, find_unmapped,)

from golden_data.logging_utils import get_normalization_logger, log_normalization_event

from golden_data.markdown_utils import df_to_markdown
from golden_data.ai.ai_taxonomist import summarize_attribute_strategy





Project root: /Users/csmiller/projects


## 2. Load Raw Data & Extract Material / Finish

We:
- Load the BigCommerce export
- Choose a product ID column
- Parse Custom Fields to extract Material and Finish
- Pivot into one row per product (`attr_pivot`)

In [2]:
# 2.1 Load raw data
raw_path = get_raw_data_path()
df_raw = pd.read_csv(raw_path, low_memory=False)

print("Loaded raw file:", raw_path)
print("Shape:", df_raw.shape)
df_raw.head()

Loaded raw file: /Users/csmiller/projects/bossard-golden-data/data/raw/bossard_raw.csv
Shape: (175917, 50)


,Item,ID,Name,Type,SKU,Options,Inventory Tracking,Current Stock,Low Stock,Price,...,Variant Image URL,Internal Image URL (Export),Image URL (Import),Image Description,Image is Thumbnail,Image Sort Order,YouTube ID,Video Title,Video Description,Video Sort Order
0,Product,3375,4049AD51H01800 - ECOGREEN ™ EMI Shielding Gask...,physical,4049AD51H01800,NaN,product,0.0,0.0,8.32,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Image,3478,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,https://cdn11.bigcommerce.com/s-mdlmxtwv57/pro...,NaN,4049AD51H01800 - ECOGREEN ™ EMI Shielding Gask...,False,0.0,NaN,NaN,NaN,NaN
2,Image,91253,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,https://cdn11.bigcommerce.com/s-mdlmxtwv57/pro...,NaN,4049AD51H01800 - ECOGREEN â„¢ EMI Shielding Ga...,False,0.0,NaN,NaN,NaN,NaN
3,Image,91258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,https://cdn11.bigcommerce.com/s-mdlmxtwv57/pro...,NaN,4049AD51H01800 - ECOGREEN â„¢ EMI Shielding Ga...,False,0.0,NaN,NaN,NaN,NaN
4,Image,91263,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,https://cdn11.bigcommerce.com/s-mdlmxtwv57/pro...,NaN,4049AD51H01800 - ECOGREEN ™ EMI Shielding Gask...,True,0.0,NaN,NaN,NaN,NaN


In [3]:
# 2.2 Identify product ID column
id_col_candidates = ["Product ID", "ID", "SKU", "Product Code/SKU"]
id_col = None
for c in id_col_candidates:
    if c in df_raw.columns:
        id_col = c
        break

if id_col is None:
    df_raw["RowID"] = df_raw.index
    id_col = "RowID"

print("Using ID column:", id_col)

# 2.3 Extract Material / Finish from Custom Fields
rows = []
for _, row in df_raw.iterrows():
    pid = row[id_col]
    for name, value in parse_cf_cell(row.get("Custom Fields")):
        if name in ("Material", "Finish") and value:
            rows.append(
                {
                    id_col: pid,
                    "Field_Name": name,
                    "Raw_Value": str(value),
                }
            )

attr_df = pd.DataFrame(rows)
print("Extracted attribute rows:", len(attr_df))
attr_df.head()

Using ID column: ID
Extracted attribute rows: 9735


,ID,Field_Name,Raw_Value
0,3375,Material,Soft Urethane
1,3376,Material,Soft Urethane
2,3378,Material,Soft Urethane
3,3379,Material,Soft Urethane
4,3380,Material,Soft Urethane


In [4]:
# 2.4 Pivot to one row per product with Material / Finish columns
attr_pivot = (
    attr_df
    .pivot_table(
        index=id_col,
        columns="Field_Name",
        values="Raw_Value",
        aggfunc="first"
    )
    .reset_index()
)

print("attr_pivot shape:", attr_pivot.shape)
attr_pivot.head()

attr_pivot shape: (6236, 3)


Field_Name,ID,Finish,Material
0,3375,NaN,Soft Urethane
1,3376,NaN,Soft Urethane
2,3378,NaN,Soft Urethane
3,3379,NaN,Soft Urethane
4,3380,NaN,Soft Urethane


## 3. Load Mapping Tables & Evaluate Coverage

We:
- Load `material_mapping.csv` and `finish_mapping.csv`
- Compute coverage
- List currently unmapped values for review

In [5]:
from golden_data.normalization.normalization import get_mapping_path

print(get_mapping_path("Material"))
print(get_mapping_path("Finish"))

/Users/csmiller/projects/bossard-golden-data/mappings/material_mapping.csv
/Users/csmiller/projects/bossard-golden-data/mappings/finish_mapping.csv


In [6]:
import pandas as pd
from golden_data.normalization.normalization import get_mapping_path
from golden_data.normalization.mapping_utils import (
    apply_mapping,
    mapping_coverage_report,
    find_unmapped,
)

material_mapping_path = get_mapping_path("Material")
finish_mapping_path = get_mapping_path("Finish")

material_mapping = pd.read_csv(material_mapping_path)
finish_mapping = pd.read_csv(finish_mapping_path)

print("Material mapping path:", material_mapping_path)
display(material_mapping.head())

print("Finish mapping path:", finish_mapping_path)
display(finish_mapping.head())

Material mapping path: /Users/csmiller/projects/bossard-golden-data/mappings/material_mapping.csv


,raw_value,normalized_material,notes
0,Steel,Steel,NaN
1,300 SERIES STAINLESS STEEL,300 SERIES STAINLESS STEEL,NaN
2,Stainless Steel,Stainless Steel,NaN
3,Aluminum,Aluminum,NaN
4,Nylon 6/6,Nylon 6/6,NaN


Finish mapping path: /Users/csmiller/projects/bossard-golden-data/mappings/finish_mapping.csv


,raw_value,normalized_finish,notes
0,Passivated and/or tested per ASTM A380,Passivated and/or tested per ASTM A380,NaN
1,Natural,Natural,NaN
2,"Zinc Plate, Bright chromate","Zinc Plate, Bright chromate",NaN
3,Powder Coat,Powder Coat,NaN
4,Passivated,Passivated,NaN


In [7]:
material_cov = mapping_coverage_report(
    series=attr_pivot["Material"],
    mapping_df=material_mapping,
    mapping_raw_col="raw_value",
    normalized_col="normalized_material",
)
finish_cov = mapping_coverage_report(
    series=attr_pivot["Finish"],
    mapping_df=finish_mapping,
    mapping_raw_col="raw_value",
    normalized_col="normalized_finish",
)

print("Material coverage:")
display(material_cov)
print("Finish coverage:")
display(finish_cov)

unmapped_material = find_unmapped(
    series=attr_pivot["Material"],
    mapping_df=material_mapping,
)
unmapped_finish = find_unmapped(
    series=attr_pivot["Finish"],
    mapping_df=finish_mapping,
)

print("Sample unmapped Material:")
display(pd.DataFrame({"Material_Unmapped": unmapped_material[:50]}))

print("Sample unmapped Finish:")
display(pd.DataFrame({"Finish_Unmapped": unmapped_finish[:50]}))

Material coverage:


,total_values,unique_values,mapped_unique_values,unmapped_unique_values,coverage_percent
0,6174,205,50,155,24.39


Finish coverage:


,total_values,unique_values,mapped_unique_values,unmapped_unique_values,coverage_percent
0,3557,137,50,87,36.5


Sample unmapped Material:


,Material_Unmapped
20,Acrylic
21,Polyurethane (Backing)
33,"Metal, Acrylic, Polycarbonate and ABS Plastic"
38,Acrylic Foam
46,"Precision Shaped Ceramic Grain (Abrasive), YF-..."
49,Copper Alloy (Contact)
50,Polyphenylene Ether and Polystyrene (Housing)
52,Polyamide 6/6 (Housing)
53,"Thermoplastic Polyester (Housing), Copper Allo..."
99,ASTM D4066 PA133 Nylon 6/6


Sample unmapped Finish:


,Finish_Unmapped
49,Gold Plated (Contact)
50,Black (Housing)
52,Natural (Housing)
53,"Blue (Housing), Gold Plated (Contact)"
78,Blue (Housing)
127,Red
132,"Bluish, Tinned, Chrome Plated"
227,Clear Chromate
243,Aluminum
414,Black Zinc Plated


In [8]:
from golden_data.analysis.coverage import attribute_health_report

field_to_mapping = {
    "Material": material_mapping,
    "Finish": finish_mapping,
}

health_df = attribute_health_report(
    df=attr_pivot,
    field_to_mapping=field_to_mapping,
    id_col=id_col,
)

display(health_df)

,Attribute,NonNull_Count,NonNull_Percent,Unique_Values,Mapped_Unique_Values,Unmapped_Unique_Values,Mapping_Coverage_Percent
0,Finish,3557,57.04,137,50.0,87.0,36.50
1,Material,6174,99.01,205,50.0,155.0,24.39
